<a href="https://colab.research.google.com/github/Sanjidh090/FATIMA_Felloship_Fall26/blob/main/CPU_inference_asr_benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# To-do:
1. Load Nemo model on CPU
2. Load 50 audios
3. Run Streaming inferences
4. Benchmarking

During an ASR competition of Neurips I realised I dont have a fixed model directory That I can use ( Suitable for my phone or cpu or edge devices ). Most of the ASR measured RTF, RTFx ,WER, TTFT measured on H100 or high end devices.

ASR seems a kind of task everybody might need everyday. GPUs or server side inference cant be always possible or secured.

To solve this issue we require a benchmark for streaming inferences score. That'd precisely tell us what model we will need.

This will be like the OpenASR leaderboard, I wish they just included this section.

Therefore we can select the best model for inference and do task specific finetuning.

This is me brainstorming the possible things we wanna do. Obviously I will vibe code and some of the stuffs I look up from documentation everytime.

1. Load model
2. Load Dataset
3. Retrieve Audio
4. Streaming Check
5. Benchmark (WER, TTFT, RTF, Speed etc)



In [ ]:
!pip install -q -U "transformers>=5.13.0" accelerate jiwer librosa soundfile pandas

In [ ]:
import os
import gc
import time
import threading
import traceback
import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoProcessor,
    AutoModelForRNNT,
    TextIteratorStreamer,
)

MODEL_ID = "nvidia/nemotron-speech-streaming-en-0.6b"

# CPU-only benchmark
DEVICE = "cpu"

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = AutoModelForRNNT.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
    device_map="cpu",
).eval()

print("Model:", MODEL_ID)
print("Device:", next(model.parameters()).device)
print("Dtype:", next(model.parameters()).dtype)
print("CPU threads:", torch.get_num_threads())
print("Sample rate:", processor.feature_extractor.sampling_rate)
print("Supported lookahead latencies:",
      processor.supported_streaming_latencies_ms)

In [ ]:
!pip install -q datasets

from datasets import load_dataset, Audio

ds = load_dataset("abnerh/TORGO-database", split="train")

print(ds)
print(ds.features)
print(ds[0].keys())

In [ ]:
import pandas as pd
import numpy as np
import os

SEED = 42
N_PAIRS = 25

# Extract metadata without copying the audio into the DataFrame.
meta = ds.remove_columns("audio").to_pandas()
meta["row"] = np.arange(len(meta))

meta["transcription"] = meta["transcription"].fillna("").astype(str).str.strip()
meta["speech_status"] = meta["speech_status"].astype(str).str.lower().str.strip()
meta["n_words"] = meta["transcription"].str.split().str.len()
meta["duration"] = pd.to_numeric(meta["duration"], errors="coerce")
meta["match_text"] = meta["transcription"].str.lower().str.strip()

# Select usable recordings.
eligible = meta[
    meta["speech_status"].isin(["dysarthria", "healthy"])
    & meta["n_words"].ge(4)
    & meta["duration"].between(1, 30)
    & meta["transcription"].ne("")
].copy()

# Find transcripts appearing in both groups.
dysarthric_texts = set(
    eligible.loc[
        eligible["speech_status"] == "dysarthria", "match_text"
    ]
)

healthy_texts = set(
    eligible.loc[
        eligible["speech_status"] == "healthy", "match_text"
    ]
)

shared_texts = sorted(dysarthric_texts & healthy_texts)

print("Eligible recordings:", len(eligible))
print("Shared transcripts:", len(shared_texts))

if len(shared_texts) < N_PAIRS:
    raise ValueError(
        f"Only {len(shared_texts)} matched transcripts found. "
        "Reduce N_PAIRS or adjust the filters."
    )

rng = np.random.default_rng(SEED)
chosen_texts = rng.choice(shared_texts, size=N_PAIRS, replace=False)

selected = []

for text in chosen_texts:
    for status in ["dysarthria", "healthy"]:
        candidates = eligible[
            (eligible["match_text"] == text)
            & (eligible["speech_status"] == status)
        ]

        selected_row = candidates.iloc[
            rng.integers(0, len(candidates))
        ]

        selected.append(selected_row)

manifest = pd.DataFrame(selected).reset_index(drop=True)

manifest = manifest[
    ["row", "speech_status", "gender", "duration", "transcription"]
]

MANIFEST_PATH = "/content/manifest.csv"
manifest.to_csv(MANIFEST_PATH, index=False)

print("\nManifest saved:", MANIFEST_PATH)
print("Total clips:", len(manifest))
print(manifest["speech_status"].value_counts())

display(manifest.head(10))

In [ ]:
assert len(manifest) == 50
assert manifest["row"].between(0, len(ds) - 1).all()
assert os.path.exists("/content/manifest.csv")

print("Dataset and manifest are ready.")
print("Audio dataset rows:", len(ds))
print("Benchmark clips:", len(manifest))
print("Transcript column: transcription")
print("Audio column: audio")

In [ ]:
import librosa

TARGET_SR = processor.feature_extractor.sampling_rate

def get_audio_from_dataset(dataset_row):
    audio_obj = ds[int(dataset_row)]["audio"]

    # Newer Hugging Face audio decoder objects
    if hasattr(audio_obj, "get_all_samples"):
        samples = audio_obj.get_all_samples()
        audio = samples.data
        sr = int(samples.sample_rate)

        if isinstance(audio, torch.Tensor):
            audio = audio.cpu().numpy()

        audio = np.asarray(audio, dtype=np.float32)

        # Convert channel-first multichannel audio to mono
        if audio.ndim == 2:
            audio = audio.mean(axis=0) if audio.shape[0] <= 8 else audio.mean(axis=1)

    # Older Hugging Face Audio dictionaries
    elif isinstance(audio_obj, dict):
        sr = int(audio_obj.get("sampling_rate", TARGET_SR))
        audio = audio_obj.get("array")

        if audio is None and audio_obj.get("path"):
            import soundfile as sf
            audio, sr = sf.read(audio_obj["path"], dtype="float32")

        audio = np.asarray(audio, dtype=np.float32)

        if audio.ndim == 2:
            audio = audio.mean(axis=1)

    else:
        raise TypeError(
            f"Unsupported audio object: {type(audio_obj)}"
        )

    audio = np.squeeze(audio).astype(np.float32)

    if audio.ndim != 1 or len(audio) == 0:
        raise ValueError(f"Invalid audio shape: {audio.shape}")

    if sr != TARGET_SR:
        audio = librosa.resample(
            audio,
            orig_sr=sr,
            target_sr=TARGET_SR,
        ).astype(np.float32)

    return audio, TARGET_SR


# Test one clip before running the benchmark.
test_audio, test_sr = get_audio_from_dataset(manifest.iloc[0]["row"])

print("Audio samples:", len(test_audio))
print("Sample rate:", test_sr)
print("Duration:", round(len(test_audio) / test_sr, 3), "seconds")

In [ ]:
print("Transformers version:", __import__("transformers").__version__)
print("Audio samples:", len(test_audio))
print("First chunk samples:", processor.num_samples_first_audio_chunk)
print("Subsequent chunk samples:", processor.num_samples_per_audio_chunk)
print("First chunk mel frames:", processor.num_mel_frames_first_audio_chunk)
print("Subsequent chunk mel frames:", processor.num_mel_frames_per_audio_chunk)
print("Lookahead settings:", processor.supported_num_lookahead_tokens)

In [ ]:
!pip install -q -U "transformers>=5.20.1" accelerate

import torch
import transformers
from transformers import AutoProcessor, AutoModelForRNNT

MODEL_ID = "nvidia/nemotron-3.5-asr-streaming-0.6b"

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForRNNT.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
).to("cpu").eval()

torch.set_num_threads(1)

print("Transformers:", transformers.__version__)
print("Model class:", type(model).__module__)
print("Processor class:", type(processor).__module__)
print("Model:", MODEL_ID)
print("Sample rate:", processor.feature_extractor.sampling_rate)
print("First chunk samples:", processor.num_samples_first_audio_chunk)
print("Next chunk samples:", processor.num_samples_per_audio_chunk)
print("First chunk mel frames:", processor.num_mel_frames_first_audio_chunk)
print("Next chunk mel frames:", processor.num_mel_frames_per_audio_chunk)
print("Supported lookahead:", processor.supported_num_lookahead_tokens)

In [ ]:
import time
import threading
import traceback
import numpy as np
import torch

from transformers import TextIteratorStreamer

def streaming_transcribe(audio, lookahead):
    processor.set_num_lookahead_tokens(int(lookahead))

    sr = processor.feature_extractor.sampling_rate
    audio = np.asarray(audio, dtype=np.float32).reshape(-1)

    if len(audio) == 0:
        raise ValueError("Audio is empty.")

    t_start = time.perf_counter()

    # Prepare the first chunk using the documented API.
    first_chunk_inputs = processor(
        audio[:processor.num_samples_first_audio_chunk],
        sampling_rate=sr,
        is_streaming=True,
        is_first_audio_chunk=True,
        language="en-US",
        return_tensors="pt",
    ).to(model.device, dtype=model.dtype)

    def input_features_generator():
        yield first_chunk_inputs.input_features[
            :, :processor.num_mel_frames_first_audio_chunk, :
        ]

        mel_frame_idx = processor.num_mel_frames_first_audio_chunk
        hop_length = processor.feature_extractor.hop_length
        n_fft = processor.feature_extractor.n_fft

        start_idx = mel_frame_idx * hop_length - n_fft // 2

        while True:
            end_idx = start_idx + processor.num_samples_per_audio_chunk

            # Match the official example's boundary condition.
            if end_idx >= len(audio):
                break

            if start_idx < 0 or end_idx <= start_idx:
                raise ValueError(
                    f"Invalid audio chunk boundaries: "
                    f"{start_idx}:{end_idx}"
                )

            chunk_inputs = processor(
                audio[start_idx:end_idx],
                sampling_rate=sr,
                is_streaming=True,
                is_first_audio_chunk=False,
                language="en-US",
                return_tensors="pt",
            ).to(model.device, dtype=model.dtype)

            yield chunk_inputs.input_features

            mel_frame_idx += processor.num_mel_frames_per_audio_chunk
            start_idx = mel_frame_idx * hop_length - n_fft // 2

    streamer = TextIteratorStreamer(
        processor.tokenizer,
        skip_special_tokens=True,
        timeout=300,
    )

    errors = []

    def worker():
        try:
            with torch.inference_mode():
                # The first-chunk inputs already carry the lookahead value.
                generate_kwargs = {
                    **first_chunk_inputs,
                    "input_features": input_features_generator(),
                    "streamer": streamer,
                    "max_new_tokens": 512,
                }
                model.generate(**generate_kwargs)

        except Exception:
            errors.append(traceback.format_exc())
            streamer.end()

    thread = threading.Thread(target=worker, daemon=True)
    thread.start()

    text_parts = []
    first_text_time = None

    for piece in streamer:
        if piece and piece.strip() and first_text_time is None:
            first_text_time = time.perf_counter() - t_start
        text_parts.append(piece)

    thread.join(timeout=300)

    if thread.is_alive():
        raise TimeoutError("Generation exceeded 300 seconds.")

    if errors:
        raise RuntimeError(errors[0])

    total_time = time.perf_counter() - t_start
    audio_duration = len(audio) / sr

    return {
        "prediction": "".join(text_parts).strip(),
        "ttft_s": first_text_time,
        "total_time_s": total_time,
        "audio_duration_s": audio_duration,
        "rtf": total_time / audio_duration,
        "lookahead": int(lookahead),
        "configured_latency_ms": processor.streaming_latency_ms,
    }

In [ ]:
smoke = streaming_transcribe(test_audio, lookahead=0)
print(smoke)

In [ ]:
import time
import threading
import traceback
import numpy as np
import torch
from transformers import TextIteratorStreamer

def streaming_transcribe(audio, lookahead):
    # Capture the currently loaded processor and model.
    proc = processor
    net = model

    proc.set_num_lookahead_tokens(int(lookahead))

    sr = proc.feature_extractor.sampling_rate
    audio = np.asarray(audio, dtype=np.float32).reshape(-1)

    if len(audio) == 0:
        raise ValueError("Audio is empty.")

    # Confirm the configuration actually used by this call.
    first_samples = proc.num_samples_first_audio_chunk
    next_samples = proc.num_samples_per_audio_chunk
    first_frames = proc.num_mel_frames_first_audio_chunk

    print(
        f"Using chunks: first={first_samples}, "
        f"next={next_samples}, first_frames={first_frames}"
    )

    t_start = time.perf_counter()

    first_inputs = proc(
        audio[:first_samples],
        sampling_rate=sr,
        is_streaming=True,
        is_first_audio_chunk=True,
        language="en-US",
        return_tensors="pt",
    ).to(net.device, dtype=net.dtype)

    def input_features_generator():
        yield first_inputs.input_features[:, :first_frames, :]

        mel_frame_idx = first_frames
        hop_length = proc.feature_extractor.hop_length
        n_fft = proc.feature_extractor.n_fft

        start_idx = mel_frame_idx * hop_length - n_fft // 2

        while True:
            end_idx = start_idx + next_samples

            if end_idx >= len(audio):
                break

            # Use the official chunk boundaries without modifying them.
            chunk_inputs = proc(
                audio[start_idx:end_idx],
                sampling_rate=sr,
                is_streaming=True,
                is_first_audio_chunk=False,
                language="en-US",
                return_tensors="pt",
            ).to(net.device, dtype=net.dtype)

            yield chunk_inputs.input_features

            mel_frame_idx += proc.num_mel_frames_per_audio_chunk
            start_idx = mel_frame_idx * hop_length - n_fft // 2

    streamer = TextIteratorStreamer(
        proc.tokenizer,
        skip_special_tokens=True,
        timeout=300,
    )

    errors = []

    # Use a dictionary so input_features is replaced rather than duplicated.
    generate_kwargs = {
        **first_inputs,
        "input_features": input_features_generator(),
        "streamer": streamer,
        "max_new_tokens": 512,
    }

    def worker():
        try:
            with torch.inference_mode():
                net.generate(**generate_kwargs)
        except Exception:
            errors.append(traceback.format_exc())
            streamer.end()

    thread = threading.Thread(target=worker, daemon=True)
    thread.start()

    text_parts = []
    first_text_time = None

    for piece in streamer:
        if piece and piece.strip() and first_text_time is None:
            first_text_time = time.perf_counter() - t_start
        text_parts.append(piece)

    thread.join(timeout=300)

    if thread.is_alive():
        raise TimeoutError("Streaming generation exceeded 300 seconds.")

    if errors:
        raise RuntimeError(errors[0])

    total_time = time.perf_counter() - t_start
    duration = len(audio) / sr

    return {
        "prediction": "".join(text_parts).strip(),
        "ttft_s": first_text_time,
        "total_time_s": total_time,
        "audio_duration_s": duration,
        "rtf": total_time / duration,
        "lookahead": int(lookahead),
        "configured_latency_ms": proc.streaming_latency_ms,
    }

In [ ]:
smoke = streaming_transcribe(test_audio, lookahead=0)
print(smoke)

In [ ]:
from transformers import AutoProcessor, AutoModelForRNNT
import transformers
import torch

MODEL_ID = "nvidia/nemotron-3.5-asr-streaming-0.6b"

# Reload the processor explicitly from the same checkpoint.
processor = AutoProcessor.from_pretrained(
    MODEL_ID
)

# Confirm the processor class and chunk configuration.
print("Transformers:", transformers.__version__)
print("Processor class:", type(processor).__module__)
print("Model class:", type(model).__module__)
print("Sample rate:", processor.feature_extractor.sampling_rate)
print("First chunk samples:", processor.num_samples_first_audio_chunk)
print("Next chunk samples:", processor.num_samples_per_audio_chunk)
print("First chunk mel frames:", processor.num_mel_frames_first_audio_chunk)
print("Next chunk mel frames:", processor.num_mel_frames_per_audio_chunk)
print("Supported lookahead:", processor.supported_num_lookahead_tokens)

In [ ]:
smoke = streaming_transcribe(test_audio, lookahead=0)
print(smoke)

In [ ]:
import time
import threading
import traceback
import numpy as np
import torch
from transformers import TextIteratorStreamer

def streaming_transcribe(audio, lookahead):
    if lookahead not in [3, 6, 13]:
        raise ValueError(
            "For this initial version, use lookahead 3, 6, or 13."
        )

    processor.set_num_lookahead_tokens(lookahead)

    sr = processor.feature_extractor.sampling_rate
    audio = np.asarray(audio, dtype=np.float32).reshape(-1)

    if len(audio) == 0:
        raise ValueError("Audio is empty.")

    t0 = time.perf_counter()

    first_n = processor.num_samples_first_audio_chunk
    chunk_n = processor.num_samples_per_audio_chunk
    first_frames = processor.num_mel_frames_first_audio_chunk

    print(
        f"Lookahead={lookahead}; "
        f"first={first_n}; next={chunk_n}; "
        f"first_frames={first_frames}"
    )

    first_inputs = processor(
        audio[:first_n],
        sampling_rate=sr,
        is_streaming=True,
        is_first_audio_chunk=True,
        language="en-US",
        return_tensors="pt",
    ).to(model.device, dtype=model.dtype)

    def feature_generator():
        yield first_inputs.input_features[:, :first_frames, :]

        mel_idx = first_frames
        hop = processor.feature_extractor.hop_length
        n_fft = processor.feature_extractor.n_fft

        start = mel_idx * hop - n_fft // 2

        while start >= 0:
            end = start + chunk_n

            if end >= len(audio):
                break

            chunk = audio[start:end]
            if len(chunk) == 0:
                break

            chunk_inputs = processor(
                chunk,
                sampling_rate=sr,
                is_streaming=True,
                is_first_audio_chunk=False,
                language="en-US",
                return_tensors="pt",
            ).to(model.device, dtype=model.dtype)

            yield chunk_inputs.input_features

            mel_idx += processor.num_mel_frames_per_audio_chunk
            start = mel_idx * hop - n_fft // 2

    streamer = TextIteratorStreamer(
        processor.tokenizer,
        skip_special_tokens=True,
        timeout=300,
    )

    errors = []

    kwargs = {
        **first_inputs,
        "input_features": feature_generator(),
        "streamer": streamer,
        "max_new_tokens": 512,
    }

    def worker():
        try:
            with torch.inference_mode():
                model.generate(**kwargs)
        except Exception:
            errors.append(traceback.format_exc())
            streamer.end()

    thread = threading.Thread(target=worker, daemon=True)
    thread.start()

    pieces = []
    ttft = None

    for piece in streamer:
        if piece and piece.strip() and ttft is None:
            ttft = time.perf_counter() - t0
        pieces.append(piece)

    thread.join(timeout=300)

    if thread.is_alive():
        raise TimeoutError("Generation exceeded 300 seconds.")

    if errors:
        raise RuntimeError(errors[0])

    elapsed = time.perf_counter() - t0
    duration = len(audio) / sr

    return {
        "prediction": "".join(pieces).strip(),
        "ttft_s": ttft,
        "total_time_s": elapsed,
        "audio_duration_s": duration,
        "rtf": elapsed / duration,
        "lookahead": lookahead,
        "configured_latency_ms": processor.streaming_latency_ms,
    }

In [ ]:
smoke = streaming_transcribe(test_audio, lookahead=3)
print(smoke)

In [ ]:
smoke_results = []

for lookahead in [3, 6, 13]:
    result = streaming_transcribe(test_audio, lookahead)
    smoke_results.append(result)

display(pd.DataFrame(smoke_results))

In [ ]:
import pandas as pd
import numpy as np
from jiwer import wer, cer
from pathlib import Path

LOOKAHEADS = [3, 6, 13]
RESULT_PATH = "/content/nemotron_cpu_results.csv"

# Resume safely if a previous run already produced results.
if Path(RESULT_PATH).exists():
    results = pd.read_csv(RESULT_PATH).to_dict("records")
else:
    results = []

completed = {
    (int(r["row"]), int(r["lookahead"]))
    for r in results
    if not r.get("error")
}

for lookahead in LOOKAHEADS:
    print(f"\n=== Lookahead {lookahead} ===")

    for _, row in manifest.iterrows():
        idx = int(row["row"])

        if (idx, lookahead) in completed:
            continue

        try:
            audio, sr = get_audio_from_dataset(idx)

            out = streaming_transcribe(audio, lookahead)

            reference = str(row["transcription"])
            prediction = out["prediction"]

            result = {
                "row": idx,
                "speech_status": row["speech_status"],
                "lookahead": lookahead,
                "configured_latency_ms": out["configured_latency_ms"],
                "ttft_ms": (
                    out["ttft_s"] * 1000
                    if out["ttft_s"] is not None else np.nan
                ),
                "audio_duration_s": out["audio_duration_s"],
                "total_time_s": out["total_time_s"],
                "rtf": out["rtf"],
                "wer": wer(reference.lower(), prediction.lower()),
                "cer": cer(reference.lower(), prediction.lower()),
                "reference": reference,
                "prediction": prediction,
                "error": "",
            }

        except Exception as exc:
            result = {
                "row": idx,
                "speech_status": row["speech_status"],
                "lookahead": lookahead,
                "error": repr(exc),
            }
            print(f"Failed row {idx}: {exc}")

        results.append(result)

        # Save after every recording.
        pd.DataFrame(results).to_csv(RESULT_PATH, index=False)

    print(f"Saved results: {RESULT_PATH}")

print("\nBenchmark loop finished.")
print("Total result rows:", len(results))

In [ ]:
results_df = pd.read_csv("/content/nemotron_cpu_results.csv")

valid = results_df[
    results_df["error"].fillna("").eq("")
].copy()

summary = (
    valid.groupby(["lookahead", "speech_status"])
    .agg(
        clips=("row", "nunique"),
        median_TTFT_ms=("ttft_ms", "median"),
        p95_TTFT_ms=("ttft_ms", lambda x: x.quantile(0.95)),
        mean_RTF=("rtf", "mean"),
        mean_WER=("wer", "mean"),
        mean_CER=("cer", "mean"),
    )
    .reset_index()
)

display(summary.round(4))

summary.to_csv(
    "/content/nemotron_cpu_summary.csv",
    index=False,
)

print("Summary saved.")

Dysarthric speeches are a challanging test for ASR models. Streaming models goes even worse than the normal models which passes on recorded audios.

Howeever, I found Zipformer model is super fast and surprisingly moonshine streaming model is slow, too. The next results will speak fro themselves.

In [ ]:
# Moonshine Tiny Streaming

In [ ]:
!pip -q install -U moonshine-voice jiwer soundfile

In [ ]:
import subprocess
import re
from pathlib import Path

cmd = [
    "moonshine-voice", "download",
    "--stt", "--language", "en",
    "--model-arch", "2",
]

download = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    check=True,
)

print(download.stdout)
print(download.stderr[-2000:] if download.stderr else "")

output = download.stdout + "\n" + download.stderr

match = re.search(
    r"Downloaded model path:\s*(.+)",
    output,
)

if not match:
    raise RuntimeError(
        "Could not identify the model directory. "
        "Read the downloader output above and set MODEL_PATH manually."
    )

MODEL_PATH = match.group(1).strip()
print("Moonshine model directory:", MODEL_PATH)
print("Directory exists:", Path(MODEL_PATH).is_dir())

In [ ]:
import subprocess
import re
from pathlib import Path

cmd = [
    "moonshine-voice", "download",
    "--stt", "--language", "en",
    "--model-arch", "2",
]

download = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    check=True,
)

print(download.stdout)
print(download.stderr[-2000:] if download.stderr else "")

output = download.stdout + "\n" + download.stderr

match = re.search(
    r"Downloaded model path:\s*(.+)",
    output,
)

if not match:
    raise RuntimeError(
        "Could not identify the model directory. "
        "Read the downloader output above and set MODEL_PATH manually."
    )

MODEL_PATH = match.group(1).strip()
print("Moonshine model directory:", MODEL_PATH)
print("Directory exists:", Path(MODEL_PATH).is_dir())

In [ ]:
import time
import numpy as np
import threading

from moonshine_voice import Transcriber, ModelArch
from moonshine_voice import TranscriptEventListener

class MoonshineListener(TranscriptEventListener):
    def __init__(self):
        self.first_text_time = None
        self.start_time = None
        self.completed_lines = []
        self.latest_text = ""

    def on_line_text_changed(self, event):
        text = event.line.text or ""

        if text.strip():
            self.latest_text = text

            if self.first_text_time is None:
                self.first_text_time = (
                    time.perf_counter() - self.start_time
                )

    def on_line_completed(self, event):
        text = event.line.text or ""

        if text.strip():
            self.completed_lines.append(text)


def moonshine_transcribe(audio, sample_rate=16000,
                         chunk_ms=100, simulate_realtime=True):

    audio = np.asarray(audio, dtype=np.float32).reshape(-1)

    if len(audio) == 0:
        raise ValueError("Audio is empty.")

    listener = MoonshineListener()

    transcriber = Transcriber(
        model_path=MODEL_PATH,
        model_arch=ModelArch.TINY_STREAMING,
        update_interval=0.1,
    )

    transcriber.add_listener(listener)

    try:
        transcriber.start()

        start = time.perf_counter()
        listener.start_time = start

        chunk_size = int(sample_rate * chunk_ms / 1000)

        for i in range(0, len(audio), chunk_size):
            chunk = audio[i:i + chunk_size]

            transcriber.add_audio(chunk, sample_rate)

            if simulate_realtime:
                # Simulate audio arriving at its natural rate.
                time.sleep(len(chunk) / sample_rate)

        transcriber.stop()

        elapsed = time.perf_counter() - start
        duration = len(audio) / sample_rate

        return {
            "prediction": " ".join(listener.completed_lines).strip(),
            "partial_text": listener.latest_text,
            "ttft_s": listener.first_text_time,
            "total_time_s": elapsed,
            "audio_duration_s": duration,
            "rtf": elapsed / duration,
        }

    finally:
        transcriber.close()

In [ ]:
moonshine_smoke = moonshine_transcribe(
    test_audio,
    sample_rate=16000,
    chunk_ms=100,
    simulate_realtime=True,
)

for key, value in moonshine_smoke.items():
    print(f"{key}: {value}")

In [ ]:
import pandas as pd
import numpy as np
from jiwer import wer, cer

MOONSHINE_RESULTS = "/content/moonshine_tiny_streaming_results.csv"

moonshine_results = []

for _, row in manifest.iterrows():
    idx = int(row["row"])

    try:
        audio, sr = get_audio_from_dataset(idx)

        result = moonshine_transcribe(
            audio,
            sample_rate=sr,
            chunk_ms=100,
            simulate_realtime=True,
        )

        reference = str(row["transcription"])
        prediction = result["prediction"]

        record = {
            "row": idx,
            "speech_status": row["speech_status"],
            "audio_duration_s": result["audio_duration_s"],
            "ttft_ms": (
                result["ttft_s"] * 1000
                if result["ttft_s"] is not None else np.nan
            ),
            "total_time_s": result["total_time_s"],
            "rtf": result["rtf"],
            "wer": wer(reference.lower(), prediction.lower()),
            "cer": cer(reference.lower(), prediction.lower()),
            "reference": reference,
            "prediction": prediction,
            "error": "",
        }

    except Exception as exc:
        record = {
            "row": idx,
            "speech_status": row["speech_status"],
            "error": repr(exc),
        }
        print(f"Failed row {idx}: {exc}")

    moonshine_results.append(record)

    pd.DataFrame(moonshine_results).to_csv(
        MOONSHINE_RESULTS,
        index=False,
    )

print("Saved:", MOONSHINE_RESULTS)
print("Completed recordings:", len(moonshine_results))

In [ ]:
moonshine_df = pd.read_csv(MOONSHINE_RESULTS)

valid = moonshine_df[
    moonshine_df["error"].fillna("").eq("")
].copy()

moonshine_summary = (
    valid.groupby("speech_status")
    .agg(
        clips=("row", "nunique"),
        median_TTFT_ms=("ttft_ms", "median"),
        p95_TTFT_ms=("ttft_ms", lambda x: x.quantile(0.95)),
        mean_RTF=("rtf", "mean"),
        mean_WER=("wer", "mean"),
        mean_CER=("cer", "mean"),
    )
    .reset_index()
)

display(moonshine_summary.round(4))

moonshine_summary.to_csv(
    "/content/moonshine_tiny_summary.csv",
    index=False,
)

In [ ]:
#zipformer

In [ ]:
!pip -q install -U sherpa-onnx jiwer soundfile

import os
import glob
import tarfile
import subprocess
from pathlib import Path

ZIPFORMER_DIR = "/content/zipformer_streaming"
os.makedirs(ZIPFORMER_DIR, exist_ok=True)

archive = "/content/zipformer_streaming.tar.bz2"
url = (
    "https://github.com/k2-fsa/sherpa-onnx/releases/download/"
    "asr-models/sherpa-onnx-streaming-zipformer-en-2023-06-26.tar.bz2"
)

if not glob.glob(ZIPFORMER_DIR + "/**/encoder-*.onnx", recursive=True):
    subprocess.run(
        ["wget", "-q", "-O", archive, url],
        check=True,
    )

    with tarfile.open(archive, "r:bz2") as tar:
        tar.extractall(ZIPFORMER_DIR, filter="data")

    os.remove(archive)

print("Downloaded model files:")
for p in glob.glob(ZIPFORMER_DIR + "/**/*", recursive=True):
    if os.path.isfile(p) and (
        p.endswith(".onnx") or p.endswith("tokens.txt")
    ):
        print(p)

In [ ]:
import sherpa_onnx

model_root = next(
    Path(ZIPFORMER_DIR).rglob("tokens.txt")
).parent

def find_model(pattern):
    matches = list(model_root.glob(pattern))
    if not matches:
        raise FileNotFoundError(
            f"Cannot find {pattern} in {model_root}"
        )
    # Prefer the INT8 version for CPU inference.
    int8 = [p for p in matches if ".int8.onnx" in p.name]
    return str(int8[0] if int8 else matches[0])

encoder = find_model("encoder-*.onnx")
decoder = find_model("decoder-*.onnx")
joiner = find_model("joiner-*.onnx")
tokens = str(model_root / "tokens.txt")

recognizer = sherpa_onnx.OnlineRecognizer.from_transducer(
    tokens=tokens,
    encoder=encoder,
    decoder=decoder,
    joiner=joiner,
    num_threads=1,
    sample_rate=16000,
    feature_dim=80,
    decoding_method="greedy_search",
)

print("Zipformer streaming recognizer loaded.")
print("Encoder:", encoder)
print("Decoder:", decoder)
print("Joiner:", joiner)

In [ ]:
import time
import numpy as np
import pandas as pd
from jiwer import wer, cer

def zipformer_transcribe(audio, sr=16000, chunk_ms=100):
    audio = np.asarray(audio, dtype=np.float32).reshape(-1)

    if sr != 16000:
        raise ValueError("Resample audio to 16 kHz before inference.")
    if len(audio) == 0:
        raise ValueError("Empty audio.")

    stream = recognizer.create_stream()
    chunk_size = int(sr * chunk_ms / 1000)

    t0 = time.perf_counter()
    ttft = None
    previous_text = ""

    for start in range(0, len(audio), chunk_size):
        chunk = audio[start:start + chunk_size]
        stream.accept_waveform(sr, chunk)

        while recognizer.is_ready(stream):
            recognizer.decode_stream(stream)

        current_text = recognizer.get_result(stream).strip()

        if current_text and ttft is None:
            ttft = time.perf_counter() - t0

        previous_text = current_text

    stream.input_finished()

    while recognizer.is_ready(stream):
        recognizer.decode_stream(stream)

    final_text = recognizer.get_result(stream).strip()
    elapsed = time.perf_counter() - t0
    duration = len(audio) / sr

    return {
        "prediction": final_text,
        "ttft_s": ttft,
        "total_time_s": elapsed,
        "audio_duration_s": duration,
        "rtf": elapsed / duration,
    }

# Test one of your existing recordings.
zip_smoke = zipformer_transcribe(test_audio)

print(zip_smoke)

In [ ]:
ZIP_RESULTS = "/content/zipformer_cpu_results.csv"
zip_rows = []

for _, row in manifest.iterrows():
    idx = int(row["row"])

    try:
        audio, sr = get_audio_from_dataset(idx)

        if sr != 16000:
            import librosa
            audio = librosa.resample(
                audio, orig_sr=sr, target_sr=16000
            ).astype(np.float32)

        out = zipformer_transcribe(audio, sr=16000)
        reference = str(row["transcription"])
        prediction = out["prediction"]

        record = {
            "row": idx,
            "speech_status": row["speech_status"],
            "ttft_ms": (
                out["ttft_s"] * 1000
                if out["ttft_s"] is not None else np.nan
            ),
            "audio_duration_s": out["audio_duration_s"],
            "total_time_s": out["total_time_s"],
            "rtf": out["rtf"],
            "wer": wer(reference.lower(), prediction.lower()),
            "cer": cer(reference.lower(), prediction.lower()),
            "reference": reference,
            "prediction": prediction,
            "error": "",
        }

    except Exception as exc:
        record = {
            "row": idx,
            "speech_status": row["speech_status"],
            "error": repr(exc),
        }
        print(f"Failed row {idx}: {exc}")

    zip_rows.append(record)
    pd.DataFrame(zip_rows).to_csv(ZIP_RESULTS, index=False)

print("Saved:", ZIP_RESULTS)
display(pd.DataFrame(zip_rows).head())

In [ ]:
# Parakeet

In [ ]:
import gc
import torch
import time
import numpy as np
import pandas as pd

from transformers import AutoProcessor, AutoModelForCTC

# Release Nemotron if it is no longer needed in memory.
if "model" in globals():
    del model
if "processor" in globals():
    del processor

gc.collect()

PARAKEET_ID = "nvidia/parakeet-ctc-0.6b"

parakeet_processor = AutoProcessor.from_pretrained(PARAKEET_ID)
parakeet_model = AutoModelForCTC.from_pretrained(
    PARAKEET_ID,
    torch_dtype=torch.float32,
).to("cpu").eval()

torch.set_num_threads(1)

print("Model:", PARAKEET_ID)
print("Model class:", type(parakeet_model).__name__)
print("Sample rate:", parakeet_processor.feature_extractor.sampling_rate)

In [ ]:
def parakeet_transcribe(audio, sr=16000):
    import librosa

    audio = np.asarray(audio, dtype=np.float32).reshape(-1)

    if sr != 16000:
        audio = librosa.resample(
            audio,
            orig_sr=sr,
            target_sr=16000,
        ).astype(np.float32)

    if len(audio) == 0:
        raise ValueError("Audio is empty.")

    start = time.perf_counter()

    inputs = parakeet_processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt",
    )

    with torch.inference_mode():
        outputs = parakeet_model(**inputs)
        predicted_ids = outputs.logits.argmax(dim=-1)
        prediction = parakeet_processor.batch_decode(
            predicted_ids
        )[0]

    elapsed = time.perf_counter() - start
    duration = len(audio) / 16000

    return {
        "prediction": prediction.strip(),
        "total_time_s": elapsed,
        "audio_duration_s": duration,
        "rtf": elapsed / duration,
    }

parakeet_smoke = parakeet_transcribe(test_audio, sr=16000)
print(parakeet_smoke)

In [ ]:
from jiwer import wer, cer

PARAKEET_RESULTS = "/content/parakeet_ctc_cpu_results.csv"
parakeet_rows = []

for _, row in manifest.iterrows():
    idx = int(row["row"])

    try:
        audio, sr = get_audio_from_dataset(idx)
        out = parakeet_transcribe(audio, sr=sr)

        reference = str(row["transcription"])
        prediction = out["prediction"]

        record = {
            "row": idx,
            "speech_status": row["speech_status"],
            "total_time_s": out["total_time_s"],
            "audio_duration_s": out["audio_duration_s"],
            "rtf": out["rtf"],
            "wer": wer(reference.lower(), prediction.lower()),
            "cer": cer(reference.lower(), prediction.lower()),
            "reference": reference,
            "prediction": prediction,
            "error": "",
        }

    except Exception as exc:
        record = {
            "row": idx,
            "speech_status": row["speech_status"],
            "error": repr(exc),
        }
        print(f"Failed row {idx}: {exc}")

    parakeet_rows.append(record)
    pd.DataFrame(parakeet_rows).to_csv(
        PARAKEET_RESULTS, index=False
    )

print("Saved:", PARAKEET_RESULTS)

In [ ]:
import os
import numpy as np
import pandas as pd
from jiwer import wer, cer
from IPython.display import display

FILES = {
    "Nemotron 3.5 0.6B": "/content/nemotron_cpu_results.csv",
    "Zipformer Streaming": "/content/zipformer_cpu_results.csv",
    "Parakeet CTC 0.6B": "/content/parakeet_ctc_cpu_results.csv",
    "Moonshine Tiny Streaming": "/content/moonshine_tiny_streaming_results.csv",
}

def normalize_text(text):
    import re
    text = str(text).lower()
    text = re.sub(r"[^\w\s']", " ", text)
    return " ".join(text.split())

def safe_mean(series):
    return pd.to_numeric(series, errors="coerce").mean()

def safe_median(series):
    return pd.to_numeric(series, errors="coerce").median()

summary_rows = []
all_results = []

for model_name, filepath in FILES.items():
    if not os.path.exists(filepath):
        print(f"Not found (skipped): {filepath}")
        continue

    df = pd.read_csv(filepath)
    df["model"] = model_name

    if "error" not in df.columns:
        df["error"] = ""

    df["error"] = df["error"].fillna("").astype(str)
    valid = df[df["error"].str.strip().eq("")].copy()

    if valid.empty:
        print(f"No successful recordings: {model_name}")
        continue

    # Convert timing columns to numeric.
    for col in [
        "ttft_ms", "total_time_s", "audio_duration_s",
        "duration_s", "rtf", "wer", "cer"
    ]:
        if col in valid.columns:
            valid[col] = pd.to_numeric(valid[col], errors="coerce")

    # Standardize audio duration column.
    if "audio_duration_s" not in valid.columns:
        if "duration_s" in valid.columns:
            valid["audio_duration_s"] = valid["duration_s"]
        else:
            valid["audio_duration_s"] = np.nan

    # Derive RTF if missing.
    if "rtf" not in valid.columns:
        valid["rtf"] = (
            valid["total_time_s"] / valid["audio_duration_s"]
        )

    # Recalculate text metrics consistently when transcripts exist.
    if {"reference", "prediction"}.issubset(valid.columns):
        valid["ref_norm"] = valid["reference"].map(normalize_text)
        valid["pred_norm"] = valid["prediction"].map(normalize_text)

        valid["utterance_wer"] = [
            wer(r, p)
            for r, p in zip(valid["ref_norm"], valid["pred_norm"])
        ]
        valid["utterance_cer"] = [
            cer(r, p)
            for r, p in zip(valid["ref_norm"], valid["pred_norm"])
        ]

        corpus_wer = wer(
            valid["ref_norm"].tolist(),
            valid["pred_norm"].tolist()
        )
        corpus_cer = cer(
            valid["ref_norm"].tolist(),
            valid["pred_norm"].tolist()
        )
    else:
        corpus_wer = safe_mean(valid["wer"]) if "wer" in valid else np.nan
        corpus_cer = safe_mean(valid["cer"]) if "cer" in valid else np.nan

    # Keep Nemotron configurations separate.
    if "lookahead" in valid.columns:
        groups = valid.groupby("lookahead", dropna=False)
    else:
        groups = [(None, valid)]

    for lookahead, group in groups:
        duration = group["audio_duration_s"].sum()
        elapsed = group["total_time_s"].sum() if "total_time_s" in group else np.nan

        if {"reference", "prediction"}.issubset(group.columns):
            refs = group["reference"].map(normalize_text).tolist()
            preds = group["prediction"].map(normalize_text).tolist()
            group_wer = wer(refs, preds)
            group_cer = cer(refs, preds)
        else:
            group_wer = safe_mean(group["wer"]) if "wer" in group else np.nan
            group_cer = safe_mean(group["cer"]) if "cer" in group else np.nan

        total_rtf = elapsed / duration if duration > 0 and pd.notna(elapsed) else np.nan

        summary_rows.append({
            "Model": model_name,
            "Lookahead": lookahead if lookahead is not None else "N/A",
            "Successful clips": group["row"].nunique() if "row" in group else len(group),
            "Failed runs": int(df["error"].str.strip().ne("").sum()),
            "Corpus WER (%)": group_wer * 100,
            "Corpus CER (%)": group_cer * 100,
            "Median TTFT (ms)": (
                safe_median(group["ttft_ms"])
                if "ttft_ms" in group else np.nan
            ),
            "Median RTF": safe_median(group["rtf"]),
            "Aggregate RTF": total_rtf,
            "Aggregate RTFx": 1 / total_rtf if pd.notna(total_rtf) and total_rtf > 0 else np.nan,
        })

    all_results.append(df)

comparison = pd.DataFrame(summary_rows)

if comparison.empty:
    print("No usable result files were found. Check the filenames above.")
else:
    comparison = comparison.sort_values(
        ["Model", "Lookahead"], key=lambda s: s.astype(str)
    ).reset_index(drop=True)

    display(comparison.round(3))

    comparison.to_csv(
        "/content/asr_model_comparison.csv",
        index=False
    )
    print("Saved: /content/asr_model_comparison.csv")